#CUDA_Lab01_230109015


In [ ]:
# Hardware Telemetry Inspection
!nvidia-smi

from numba import cuda

device = cuda.get_current_device()
print(f"Device Name: {device.name}")  # Removed .decode('utf-8')
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Fri Oct  9 05:03:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             29W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

# 3. Benchmark Harness
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # Measure CPU Pure Loop (skip for large N to avoid hanging)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    # GPU Execution Configuration
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip Measurement (Includes Host-to-Device and Device-to-Host transfers over PCIe)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only Measurement (Data pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

# Run Benchmark Across Orders of Magnitude
sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
print(f"{'N':<12} | {'CPU Time (ms)':<15} | {'GPU Total (ms)':<15} | {'GPU Kernel (ms)':<15} | {'Winner':<10}")
print("-" * 75)

for N in sizes:
    cpu_t, gpu_tot, gpu_kern = benchmark_run(N)
    if np.isnan(cpu_t):
        winner = "GPU"
    else:
        winner = "CPU" if cpu_t < gpu_tot else "GPU"
    print(f"{N:<12} | {cpu_t:<15.4f} | {gpu_tot:<15.4f} | {gpu_kern:<15.4f} | {winner:<10}")

N            | CPU Time (ms)   | GPU Total (ms)  | GPU Kernel (ms) | Winner    
---------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100          | 0.0772          | 1.3826          | 0.0968          | CPU       
1000         | 0.2464          | 0.5251          | 0.0571          | CPU       
10000        | 2.3327          | 0.6006          | 0.0556          | GPU       
100000       | 25.2405         | 0.6953          | 0.0721          | GPU       
1000000      | 258.9053        | 4.9447          | 0.1212          | GPU       
5000000      | nan             | 12.3212         | 0.2637          | GPU       
10000000     | nan             | 23.5384         | 0.4464          | GPU       


In [ ]:
# Task 3.1: Missing Guard Experiment
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4  # 4 * 256 = 1024 threads launched for 1000 elements

print("Launching defective kernel with 1024 threads on 1000-element array...")
try:
    defective_kernel[blocks_per_grid, threads_per_block](d_arr)
    cuda.synchronize()
    h_result = d_arr.copy_to_host()
    print(f"Execution finished without hard crash. Array size: {len(h_result)}. Last element: {h_result[-1]}")
except Exception as e:
    print(f"CUDA Error triggered: {e}")

print("\n" + "="*60 + "\n")

# Task 3.2: Spatial Thread Coordinate Inspector (N = 37)
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # 5 blocks = 40 threads

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<15}")
print("-" * 55)
for i in range(32, 40):  # Boundary transition in Block 4
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<15}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished without hard crash. Array size: 1000. Last element: 999.0


Global ID  | Block ID   | Thread ID  | Status         
-------------------------------------------------------
32         | 4          | 0          | ACTIVE         
33         | 4          | 1          | ACTIVE         
34         | 4          | 2          | ACTIVE         
35         | 4          | 3          | ACTIVE         
36         | 4          | 4          | ACTIVE         
37         | 4          | 5          | IDLE (GUARDED) 
38         | 4          | 6          | IDLE (GUARDED) 
39         | 4          | 7          | IDLE (GUARDED) 


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [ ]:
# Task 4.1: Buggy Naive Summation (Race Condition)
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

# Correct Atomic Summation
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
h_arr = np.ones(N, dtype=np.float32)
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

print("--- Testing Naive Kernel (5 Trials) ---")
for trial in range(1, 6):
    d_arr = cuda.to_device(h_arr)
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
    cuda.synchronize()
    res = d_total.copy_to_host()[0]
    dev = 50000.0 - res
    print(f"Trial {trial}: Naive Output = {res:.1f} | Deviation = {dev:.1f}")

print("\n--- Testing Atomic Kernel & Benchmarking Performance ---")

# Naive Timing
d_arr = cuda.to_device(h_arr)
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()
naive_time = (time.perf_counter() - t0) * 1000.0

# Atomic Timing
d_arr = cuda.to_device(h_arr)
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()
atomic_time = (time.perf_counter() - t0) * 1000.0
atomic_res = d_total.copy_to_host()[0]

print(f"Atomic Output: {atomic_res:.1f} (Exact match: {atomic_res == 50000.0})")
print(f"Naive Time: {naive_time:.4f} ms | Atomic Time: {atomic_time:.4f} ms")

--- Testing Naive Kernel (5 Trials) ---
Trial 1: Naive Output = 2.0 | Deviation = 49998.0
Trial 2: Naive Output = 2.0 | Deviation = 49998.0
Trial 3: Naive Output = 2.0 | Deviation = 49998.0
Trial 4: Naive Output = 2.0 | Deviation = 49998.0
Trial 5: Naive Output = 2.0 | Deviation = 49998.0

--- Testing Atomic Kernel & Benchmarking Performance ---
Atomic Output: 50000.0 (Exact match: True)
Naive Time: 0.3245 ms | Atomic Time: 48.9072 ms


In [ ]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

# 2D Execution Configuration
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()

print(f"Generated 2D matrix shape: {result_canvas.shape}")
print(f"Blocks in X: {blocks_2d[0]} | Blocks in Y: {blocks_2d[1]} | Total Blocks: {blocks_2d[0] * blocks_2d[1]}")
print(f"Center value (512, 512): {result_canvas[512, 512]:.3f}")
print(f"Outer corner value (0, 0): {result_canvas[0, 0]:.3f}")

Generated 2D matrix shape: (1024, 1024)
Blocks in X: 64 | Blocks in Y: 64 | Total Blocks: 4096
Center value (512, 512): 1.000
Outer corner value (0, 0): 0.000


In [ ]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    h.update(cuda.get_current_device().name.encode('utf-8'))  # Added .encode('utf-8')
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

In [ ]:
# Function call execution
STUDENT_ID = "230109015"
CROSSOVER_N = 10000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 6AFFCB10E57BCFAF


'6AFFCB10E57BCFAF'